# Playroom

This notebook contains examples for how to generate text using the steered model, when steering towards one or more concepts.

In [ ]:
import torch
import activation_additions as aa
from typing import Dict, Union, Callable, Tuple, List
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector, print_n_comparisons, ActivationAddition

In [ ]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

In [ ]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "tokens_to_generate": 50,
    "seed": 0, 
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about apples constantly",
        prompt2="I do not talk about apples constantly",
        coeff=2,
        act_name=16,
    )
]

COMPLETION_PROMPT = "I wanted to bring up a topic of discussion about"
print_n_comparisons(
    model=model,
    prompt=COMPLETION_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="the sun",
        prompt2=" ",
        coeff=4,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="eyeglasses",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

COMPLETION_PROMPT = "What I am going to do now is"
print_n_comparisons(
    model=model,
    prompt=COMPLETION_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)
